# OpenCV Road Following Live (No ML Model)

Notebook này sử dụng thuật toán Computer Vision truyền thống (Canny Edge Detection + Hough Transform) của OpenCV để tìm vạch kẻ đường thay vì dùng mạng nơ-ron.

Khởi tạo Camera và Xe (Sử dụng `JetRacerController` từ `basic_motion.py`)

In [1]:
import os
import time
import cv2
import numpy as np
import ipywidgets
import threading
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from basic_motion import JetRacerController

# Khởi động lại camera daemon
os.system('echo "jetson" | sudo -S systemctl restart nvargus-daemon')
time.sleep(2)

try:
    if 'camera' in globals():
        camera.running = False
        camera.unobserve_all()
except:
    pass

camera = CSICamera(width=224, height=224, capture_fps=0)

# Khởi tạo bộ điều khiển xe
car = JetRacerController()


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


[JetRacer] Mức ga (throttle): 0.00
[JetRacer] Đã dừng xe!
[JetRacer] Góc lái (steering): 0.00


Tạo giao diện hiển thị Live View

In [2]:
state_widget = ipywidgets.ToggleButtons(options=['stop', 'live'], description='Trạng thái', value='stop')
prediction_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)
prediction_widget.value = bgr8_to_jpeg(np.zeros((224, 224, 3), dtype=np.uint8))

steering_gain_slider = ipywidgets.FloatSlider(description='Steering Gain', min=0.0, max=2.0, value=1.0, step=0.05, orientation='horizontal')
throttle_slider = ipywidgets.FloatSlider(description='Throttle', min=0.0, max=0.5, value=0.15, step=0.01, orientation='horizontal')

ui_widget = ipywidgets.VBox([
    prediction_widget,
    ipywidgets.HBox([state_widget]),
    ipywidgets.HBox([steering_gain_slider, throttle_slider])
])

display(ui_widget)

Thuật toán xử lý ảnh OpenCV để tìm đường đi và điều khiển xe

In [3]:
def process_cv_lane(img):
    height, width = img.shape[:2]
    
    # 1. Cắt ROI: Lấy nửa dưới của ảnh
    roi_top = height // 2
    roi_bottom = height
    roi = img[roi_top:roi_bottom, :]
    
    # 2. Chuyển sang không gian màu HSV
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    
    # 3. Lọc màu viền sa bàn (Cam, Đỏ) và vạch trong (Trắng)
    lower_orange = np.array([5, 100, 100])
    upper_orange = np.array([25, 255, 255])
    mask_orange = cv2.inRange(hsv, lower_orange, upper_orange)
    
    lower_red = np.array([0, 100, 100])
    upper_red = np.array([5, 255, 255])
    mask_red = cv2.inRange(hsv, lower_red, upper_red)
    mask_orange = cv2.bitwise_or(mask_orange, mask_red)
    
    
    mask = mask_orange
    
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # 4. Tìm viền bằng Canny trên vùng đã Mask
    edges = cv2.Canny(mask, 50, 150)
    
    line_image = np.zeros_like(img)
    # --- THÊM: TÌM VẬT CẢN (OBSTACLE DETECTION) ---
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges_all = cv2.Canny(blurred, 50, 150)
    
    # Loại bỏ các đường biên của vạch kẻ đường (để không nhận vạch là vật cản)
    kernel_dilate = np.ones((5,5), np.uint8)
    mask_dilated = cv2.dilate(mask, kernel_dilate, iterations=2)
    edges_obs = cv2.bitwise_and(edges_all, edges_all, mask=cv2.bitwise_not(mask_dilated))
    
    # Tìm các đường viền của vật cản
    contours, _ = cv2.findContours(edges_obs, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    obstacle_detected = False
    obs_center_x = -1
    
    if contours:
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            area = w * h # Diện tích bounding box
            if 200 < area < 8000: # Lọc nhiễu, kích thước vật cản hợp lý
                obs_center_x = x + w // 2
                obs_center_y = y + h // 2
                cv2.rectangle(line_image, (x, y + roi_top), (x+w, y+h + roi_top), (0, 255, 255), 2)
                cv2.putText(line_image, "Vat can", (x, y + roi_top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                obstacle_detected = True
                break
    # ---------------------------------------------
    
    # Bỏ dòng tìm edges cũ vì đã gộp ở trên
    # 5. Phát hiện đường (Hough Lines)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=20, minLineLength=15, maxLineGap=40)
    
    left_lines = []
    right_lines = []
    
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1: continue
            
            # Phục hồi hệ trục tọa độ của ảnh gốc
            y1 += roi_top
            y2 += roi_top
            
            slope = (y2 - y1) / (x2 - x1)
            if slope < -0.2 and (x1 < width * 0.6 or x2 < width * 0.6):
                left_lines.append(line)
                cv2.line(line_image, (x1, y1), (x2, y2), (255, 0, 0), 3) # Vạch trái (Xanh dương)
            elif slope > 0.2 and (x1 > width * 0.4 or x2 > width * 0.4):
                right_lines.append(line)
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 0, 255), 3) # Vạch phải (Đỏ)

    # 6. Tính toán điểm tâm để bẻ lái
    center_x = width // 2
    center_y = height // 2 + 50
    
    left_x = []
    right_x = []
    
    if left_lines:
        for line in left_lines:
            left_x.extend([line[0][0], line[0][2]])
    if right_lines:
        for line in right_lines:
            right_x.extend([line[0][0], line[0][2]])
            
    avg_left = sum(left_x)//len(left_x) if left_x else 0
    avg_right = sum(right_x)//len(right_x) if right_x else width
    
    if left_lines and right_lines:
        center_x = (avg_left + avg_right) // 2
    elif left_lines:
        center_x = avg_left + 80 # Né vạch trái
    elif right_lines:
        center_x = avg_right - 80 # Né vạch phải

    # --- THÊM: LOGIC NÉ VẬT CẢN ---
    view_center = width // 2
    if obstacle_detected:
        if obs_center_x < view_center:
            # Vật cản bên trái -> Dịch tâm ảo sang phải để né
            center_x += 70
            cv2.putText(line_image, "Tranh Trai -> Re Phai", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        else:
            # Vật cản bên phải -> Dịch tâm ảo sang trái để né
            center_x -= 70
            cv2.putText(line_image, "Tranh Phai -> Re Trai", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    # ------------------------------

    cv2.circle(line_image, (center_x, center_y), 8, (0, 255, 0), -1)
    result = cv2.addWeighted(img, 0.8, line_image, 1.0, 0)
    
    return result, center_x

def live_update(change):
    if state_widget.value != 'live':
        return
        
    img = change['new']
    processed_img, target_x = process_cv_lane(img.copy())
    
    steering = (target_x / (camera.width / 2.0)) - 1.0
    steering = steering * steering_gain_slider.value
    steering = max(min(steering, 1.0), -1.0)
    
    car.set_steering(steering)
    car.set_throttle(throttle_slider.value)
    
    prediction_widget.value = bgr8_to_jpeg(processed_img)

def state_changed(change):
    if change['new'] == 'stop':
        car.stop()

state_widget.observe(state_changed, names='value')


Chạy Camera

In [4]:
camera.observe(live_update, names='value')
camera.running = True